# Indexação no Qdrant

Cada precedente vira um ponto com dois vetores. O vetor `denso` usa o `bge-m3`, via Ollama, e aproxima textos com o mesmo sentido mesmo quando as palavras são diferentes. O vetor `bm25` é esparso, com stemming em português, e ajuda a achar termos exatos como "de ofício" ou "Lei 13.964".

Enunciados e teses são curtos, então cada precedente é um ponto só, sem chunking.

Antes de rodar, execute `docker compose up -d` e `ollama pull bge-m3`, além dos notebooks 01 a 03.

In [1]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")  # aviso do tqdm sem ipywidgets

import json
import uuid
from pathlib import Path

import ollama
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ENTRADA = ROOT / "data/processed/precedentes.jsonl"

QDRANT_URL = "http://localhost:6333"
COLECAO = "stj_precedentes"
MODELO_DENSO = "bge-m3"
DIMENSAO_DENSO = 1024
LOTE = 32
RECRIAR = True  # False mantém a coleção e só atualiza os pontos (os ids são determinísticos)

registros = [json.loads(linha) for linha in ENTRADA.read_text(encoding="utf-8").splitlines()]
qdrant = QdrantClient(url=QDRANT_URL, timeout=120)
bm25 = SparseTextEmbedding("Qdrant/bm25", language="portuguese")
print(len(registros), "registros |", "Qdrant", qdrant.info().version)

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Fetching 18 files:   6%|▌         | 1/18 [00:00<00:08,  2.02it/s]

Fetching 18 files:  44%|████▍     | 8/18 [00:00<00:00, 16.44it/s]

Fetching 18 files:  67%|██████▋   | 12/18 [00:01<00:00, 12.00it/s]

Fetching 18 files:  94%|█████████▍| 17/18 [00:01<00:00, 12.38it/s]

Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 11.50it/s]

3045 registros | Qdrant 1.20.0


In [2]:
if RECRIAR and qdrant.collection_exists(COLECAO):
    qdrant.delete_collection(COLECAO)

if not qdrant.collection_exists(COLECAO):
    qdrant.create_collection(
        COLECAO,
        vectors_config={"denso": models.VectorParams(size=DIMENSAO_DENSO, distance=models.Distance.COSINE)},
        sparse_vectors_config={"bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)},
    )
    qdrant.create_payload_index(COLECAO, "fonte", models.PayloadSchemaType.KEYWORD)
    qdrant.create_payload_index(COLECAO, "vigente", models.PayloadSchemaType.BOOL)
    qdrant.create_payload_index(COLECAO, "tem_tese", models.PayloadSchemaType.BOOL)


def texto_indexado(registro: dict) -> str:
    """Texto usado nos vetores. Em temas com tese, a questão ajuda a recuperar pelo problema jurídico."""
    if registro["fonte"] == "precedente" and registro["tem_tese"] and registro["questao"]:
        return f"{registro['questao']}\n{registro['enunciado']}"
    return registro["enunciado"]

In [3]:
for inicio in range(0, len(registros), LOTE):
    lote = registros[inicio : inicio + LOTE]
    textos = [texto_indexado(r) for r in lote]
    densos = ollama.embed(model=MODELO_DENSO, input=textos).embeddings
    esparsos = list(bm25.embed(textos))
    qdrant.upsert(
        COLECAO,
        points=[
            models.PointStruct(
                id=str(uuid.uuid5(uuid.NAMESPACE_URL, r["id"])),
                vector={
                    "denso": denso,
                    "bm25": models.SparseVector(indices=esparso.indices.tolist(), values=esparso.values.tolist()),
                },
                payload=r,
            )
            for r, denso, esparso in zip(lote, densos, esparsos)
        ],
    )
    if (inicio // LOTE) % 20 == 0:
        print(f"{inicio + len(lote)}/{len(registros)}", flush=True)

32/3045


672/3045


1312/3045


1952/3045


2592/3045


## Verificação

In [4]:
total = qdrant.count(COLECAO, exact=True).count
assert total == len(registros), f"esperava {len(registros)} pontos, a coleção tem {total}"


def contar(**condicoes) -> int:
    filtro = models.Filter(
        must=[models.FieldCondition(key=k, match=models.MatchValue(value=v)) for k, v in condicoes.items()]
    )
    return qdrant.count(COLECAO, count_filter=filtro, exact=True).count


print("Pontos:", total)
print("Súmulas vigentes:", contar(fonte="sumula", vigente=True))
print("Súmulas canceladas ou revogadas:", contar(fonte="sumula", vigente=False))
print("Precedentes vigentes com tese:", contar(fonte="precedente", vigente=True, tem_tese=True))
print("Precedentes vigentes em andamento:", contar(fonte="precedente", vigente=True, tem_tese=False))
print("Precedentes inativos:", contar(fonte="precedente", vigente=False))

Pontos: 3045
Súmulas vigentes: 646
Súmulas canceladas ou revogadas: 30
Precedentes vigentes com tese: 1185
Precedentes vigentes em andamento: 169
Precedentes inativos: 1015
